# Data-quality masking

This example creates a simulated five-recording dataset, runs the motion-artifact masking pipeline step, and opens the interactive data-masking GUI to review the results. The GUI supports both a Jupyter session and a regular Python command-line session.

## Create a simulated dataset

A fixed random seed makes the simulated data reproducible. Each recording is independently generated from the shared random generator, with simulated motion artifacts added to make the mask demonstration useful.

In [1]:
from pyBrainAnalyzIR.testing.simData import simMotionArtifact, simdataset

dset, _truth = simdataset(
    num_files=5,
    snr=5,
    modifiers=[simMotionArtifact],
    random_seed=2026,
)
print(f"Created {len(dset.dataset)} recordings")

 94%|█████████▍| 15/16 [00:00<00:00, 3818.79it/s]

Created 5 recordings


## Run a motion-artifact mask pipeline step

The pipeline module uses the same default settings as the GUI's **Motion artifact detection** tool. It always analyzes the `amp` time series, combines subtype results conservatively into a Boolean `(time, channel)` mask, and stores the mask on each recording. All data-quality mask steps are advanced modules, so they are hidden in Pipeline Manager unless **Show Advanced Modules** is enabled.

In [2]:
import pyBrainAnalyzIR.pipelines.modules as pipeline_modules

motion_mask = pipeline_modules.data_masking.MotionArtifactDetectionMask()
motion_mask.show()
dset = motion_mask.run(dset)

[(rec.masks["motion_artifact_detection"].dims,
  rec.masks["motion_artifact_detection"].dtype)
 for rec in dset.dataset]

Motion artifact detection mask:
Options:
	t_motion     : 0.5
	t_mask       : 1.0
	stdev_thresh : 50.0
	amp_thresh   : 5.0


[(('time', 'channel'), dtype('bool')),
 (('time', 'channel'), dtype('bool')),
 (('time', 'channel'), dtype('bool')),
 (('time', 'channel'), dtype('bool')),
 (('time', 'channel'), dtype('bool'))]

## Open the data-masking GUI

Run this cell in a notebook to open the manager without blocking the notebook kernel. The generated motion masks are loaded into the applied-mask list automatically. GUI functions default to nonblocking notebook use; pass `block=True` in a standalone script to run the GUI event loop until the window closes.

In a regular Python script the manager blocks until the window closes, so no extra flag is needed:

```python
from pyBrainAnalyzIR.vis.data_masking_manager import data_masking_manager

data_masking_manager(dset)  # Opens the window and runs the Qt event loop.
```

The command-line script should be run in an environment with a working Qt display.

In [3]:
from pyBrainAnalyzIR.vis.data_masking_manager import data_masking_manager

masking_window = data_masking_manager(dset)